In [1]:
pip install pandas requests sqlalchemy pyodbc fastparquet openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import re


server = "localhost"
database = "cleaning_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
cleaning_engine = create_engine(connection_string)

server = "localhost"
database = "dw_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
dw_engine = create_engine(connection_string)


df = pd.read_sql("SELECT * FROM dbo.clean_all_financials", cleaning_engine)

df['Date'] = pd.to_datetime(df['Date'])

df_dim_date = pd.DataFrame(df['Date'].dropna().unique(), columns=['Date'])

df_dim_date['DateKey']    = df_dim_date['Date'].dt.strftime('%Y%m%d').astype(int)
df_dim_date['Year']       = df_dim_date['Date'].dt.year
df_dim_date['Quarter']    = df_dim_date['Date'].dt.quarter
df_dim_date['Month']      = df_dim_date['Date'].dt.month
df_dim_date['Month_Name'] = df_dim_date['Date'].dt.month_name() 

df_dim_date['Date'] = df_dim_date['Date'].dt.date 

df_dim_date = df_dim_date[['DateKey', 'Date', 'Year', 'Quarter', 'Month', 'Month_Name']]

df_dim_date.to_sql(
    name='Dim_Date', con=dw_engine, 
    if_exists='replace', index=False, schema='dbo'
)

with dw_engine.begin() as conn:
    conn.execute(text("ALTER TABLE dbo.Dim_Date ALTER COLUMN DateKey INT NOT NULL;"))
    conn.execute(text("ALTER TABLE dbo.Dim_Date ADD CONSTRAINT PK_Dim_Date PRIMARY KEY (DateKey);"))

print(f"Dim_Date successfully created with {len(df_dim_date)} rows and Primary Key configured.")




Dim_Date successfully created with 7 rows and Primary Key configured.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import re


server = "localhost"
database = "cleaning_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
cleaning_engine = create_engine(connection_string)

server = "localhost"
database = "dw_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
dw_engine = create_engine(connection_string)


df = pd.read_sql("SELECT * FROM dbo.clean_all_financials", cleaning_engine)

df['Date'] = pd.to_datetime(df['Date'])



df_dim_metric = df[['Metric_Name', 'Statement_Type']].drop_duplicates().reset_index(drop=True)

df_dim_metric['Metric_ID'] = df_dim_metric.index + 1

df_dim_metric = df_dim_metric[['Metric_ID', 'Metric_Name', 'Statement_Type']]

df_dim_metric.to_sql(
    name='Dim_Metric', con=dw_engine, 
    if_exists='replace', index=False, schema='dbo'
)

with dw_engine.begin() as conn:
    conn.execute(text("ALTER TABLE dbo.Dim_Metric ALTER COLUMN Metric_ID INT NOT NULL;"))
    conn.execute(text("ALTER TABLE dbo.Dim_Metric ADD CONSTRAINT PK_Dim_Metric PRIMARY KEY (Metric_ID);"))

print(f"Dim_Metric successfully created with {len(df_dim_metric)} rows and Primary Key configured.")

Dim_Metric successfully created with 225 rows and Primary Key configured.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import re


server = "localhost"
database = "cleaning_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
cleaning_engine = create_engine(connection_string)

server = "localhost"
database = "dw_genomics"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&TrustServerCertificate=yes"
)
dw_engine = create_engine(connection_string)


df = pd.read_sql("SELECT * FROM dbo.clean_all_financials", cleaning_engine)


df['Date'] = pd.to_datetime(df['Date'])
df_dim_date['Date'] = pd.to_datetime(df_dim_date['Date'])

df_fact = pd.merge(
    df, 
    df_dim_date[['Date', 'DateKey']], 
    on='Date', 
    how='left'
)

df_fact = pd.merge(
    df_fact, 
    df_dim_metric[['Metric_Name', 'Statement_Type', 'Metric_ID']], 
    on=['Metric_Name', 'Statement_Type'], 
    how='left'
)

df_fact = df_fact[['Metric_ID', 'DateKey', 'Value']]

df_fact = df_fact.dropna(subset=['Metric_ID', 'DateKey'])
df_fact['Metric_ID'] = df_fact['Metric_ID'].astype(int)
df_fact['DateKey']   = df_fact['DateKey'].astype(int)

df_fact.to_sql(
    name='Fact_Financials', 
    con=dw_engine, 
    if_exists='replace', 
    index=False, 
    schema='dbo'
)

df_dim_date['Date'] = df_dim_date['Date'].dt.date

with dw_engine.begin() as conn:
    from sqlalchemy import text # Extra safety import
    
    
    conn.execute(text("ALTER TABLE dbo.Fact_Financials ALTER COLUMN Metric_ID INT NOT NULL;"))
    conn.execute(text("ALTER TABLE dbo.Fact_Financials ALTER COLUMN DateKey INT NOT NULL;"))
    
    conn.execute(text("""
        ALTER TABLE dbo.Fact_Financials 
        ADD CONSTRAINT FK_Fact_Metric FOREIGN KEY (Metric_ID) REFERENCES dbo.Dim_Metric(Metric_ID);
    """))
    
    conn.execute(text("""
        ALTER TABLE dbo.Fact_Financials 
        ADD CONSTRAINT FK_Fact_Date FOREIGN KEY (DateKey) REFERENCES dbo.Dim_Date(DateKey);
    """))

print(f"Fact_Financials created successfully: {len(df_fact):,} rows uploaded and linked.")

Fact_Financials created successfully: 1,575 rows uploaded and linked.
